# CBR Diagnosis DBD (versi sederhana, tanpa library)
Hanya memakai Python biasa: list, dictionary, dan perulangan `for`. Tidak ada pandas, numpy, maupun matplotlib.

Siklus CBR: **Retrieve → Reuse → Revise → Retain**.

In [ ]:
# ==========================================
# DATA
# ==========================================
gejala = [
    ("G1", "Demam"), ("G2", "Mual"), ("G3", "Muntah"), ("G4", "Lemas"),
    ("G5", "Mencret"), ("G6", "Kejang"), ("G7", "Tenggorokan Sakit"),
    ("G8", "Bintik Merah"), ("G9", "Mimisan"), ("G10", "Nyeri Sendi"),
    ("G11", "Pusing"), ("G12", "Leukosit"), ("G13", "Hematokrit"), ("G14", "Trombosit"),
]

# Basis kasus: 1 = gejala ada, 0 = gejala tidak ada (urutan G1 ... G14)
basis_kasus = [
    {"kode": "DHF-I",   "grade": "I",   "vektor": [1,0,0,0,0,0,0,1,0,0,0,1,1,1]},
    {"kode": "DHF-II",  "grade": "II",  "vektor": [1,0,0,0,0,0,0,1,1,0,0,1,1,1]},
    {"kode": "DHF-III", "grade": "III", "vektor": [1,0,0,0,0,0,0,1,1,1,0,1,1,1]},
    {"kode": "DHF-IV",  "grade": "IV",  "vektor": [1,0,0,0,0,1,0,0,0,1,0,1,1,1]},
]

urutan_grade = ["I", "II", "III", "IV"]
label = {"I": "Grade I (Ringan)", "II": "Grade II (Sedang)",
         "III": "Grade III (Berat)", "IV": "Grade IV (Sangat Berat)"}
ambang_jarak = 2.5

# Tampilkan basis kasus
print("Kasus".ljust(9), "Grade".ljust(6), " ".join(k.rjust(3) for k, _ in gejala))
for kasus in basis_kasus:
    print(kasus["kode"].ljust(9), kasus["grade"].ljust(6),
          " ".join(str(v).rjust(3) for v in kasus["vektor"]))

## 1. Fungsi-fungsi CBR
- `ubah_ke_vektor`: gejala yang dipilih → daftar 0/1
- `hitung_euclidean`: jarak = akar dari jumlah (selisih)²
- `hitung_jaccard`: gejala sama ÷ gabungan gejala
- `retrieve`: bandingkan kasus baru dengan semua kasus lama
- `reuse`: tentukan diagnosis dari kasus terdekat

In [ ]:
def ubah_ke_vektor(daftar_gejala):
    vektor = []
    for kode, nama in gejala:
        if kode in daftar_gejala:
            vektor.append(1)
        else:
            vektor.append(0)
    return vektor


def hitung_euclidean(a, b):
    total = 0
    for i in range(len(a)):
        selisih = a[i] - b[i]
        total = total + selisih ** 2
    return total ** 0.5          # akar kuadrat


def hitung_jaccard(a, b):
    sama = 0
    jumlah_a = 0
    jumlah_b = 0
    for i in range(len(a)):
        if a[i] == 1 and b[i] == 1:
            sama += 1
        jumlah_a += a[i]
        jumlah_b += b[i]
    gabungan = jumlah_a + jumlah_b - sama
    if gabungan == 0:
        return 0
    return sama / gabungan


def retrieve(x, basis):
    hasil = []
    for kasus in basis:
        y = kasus["vektor"]

        sama = 0
        beda = 0
        for i in range(len(x)):
            if x[i] == 1 and y[i] == 1:
                sama += 1
            if x[i] != y[i]:
                beda += 1

        d = hitung_euclidean(x, y)
        hasil.append({
            "kode": kasus["kode"],
            "grade": kasus["grade"],
            "gejala_sama": sama,
            "gejala_beda": beda,
            "jarak": d,
            "similarity": 1 / (1 + d),
            "jaccard": hitung_jaccard(x, y),
        })

    # urutkan: gejala beda paling sedikit dulu, kalau sama urut berdasarkan grade
    hasil.sort(key=lambda r: (r["gejala_beda"], urutan_grade.index(r["grade"])))
    return hasil


def reuse(hasil, jumlah_gejala):
    # ambil kasus terdekat dari tiap grade (baris pertama tiap grade)
    per_grade = []
    sudah_ada = []
    for r in hasil:
        if r["grade"] not in sudah_ada:
            sudah_ada.append(r["grade"])
            per_grade.append(r)

    beda_terkecil = per_grade[0]["gejala_beda"]
    kandidat = []
    for r in per_grade:
        if r["gejala_beda"] == beda_terkecil:
            kandidat.append(r["grade"])

    seri = len(kandidat) > 1
    d_min = per_grade[0]["jarak"]

    peringatan = []
    if seri:
        peringatan.append("Gejala belum cukup spesifik: seri antara Grade "
                          + " dan Grade ".join(kandidat) + ", perlu konfirmasi pakar.")
    elif jumlah_gejala <= 2:
        peringatan.append("Gejala belum cukup spesifik: hanya " + str(jumlah_gejala) + " gejala dipilih.")
    if d_min > ambang_jarak:
        peringatan.append("Gejala kurang cocok dengan pola DBD.")

    diagnosis = None if seri else kandidat[0]
    return {"diagnosis": diagnosis, "seri": seri, "kandidat": kandidat,
            "d_min": d_min, "peringatan": peringatan}


def cek_dbd(daftar_gejala, basis):
    if len(daftar_gejala) == 0:
        return {"error": "Belum ada gejala yang dipilih."}
    x = ubah_ke_vektor(daftar_gejala)
    hasil = retrieve(x, basis)
    return {"hasil": hasil, "reuse": reuse(hasil, len(daftar_gejala)), "x": x}

## 2. Contoh perhitungan (Retrieve + Reuse)

In [ ]:
kasus_baru = ["G1", "G2", "G8", "G9", "G12", "G13", "G14"]
x = ubah_ke_vektor(kasus_baru)

print("Gejala kasus baru:", kasus_baru)
print("Vektor kasus baru:", x)

# --- Langkah 1: jarak Euclidean tiap kasus lama (ditampilkan bertahap) ---
for kasus in basis_kasus:
    y = kasus["vektor"]
    print("\n" + kasus["kode"])
    print("-" * 30)

    jumlah_kuadrat = 0
    for i in range(len(x)):
        selisih_kuadrat = (x[i] - y[i]) ** 2
        jumlah_kuadrat += selisih_kuadrat
        if selisih_kuadrat == 1:
            print(gejala[i][0], gejala[i][1], "-> beda (baru =", x[i], ", lama =", y[i], ")")

    print("Jumlah kuadrat selisih =", jumlah_kuadrat)
    print("Jarak (akar)           =", round(jumlah_kuadrat ** 0.5, 3))

# --- Langkah 2: ranking semua kasus ---
hasil = retrieve(x, basis_kasus)
print("\n" + "=" * 60)
print("RANKING (Retrieve)")
print("=" * 60)
print("No  Kasus     Sama  Beda  Jarak   Similarity  Jaccard")
no = 1
for r in hasil:
    print(no, " ", r["kode"].ljust(9), str(r["gejala_sama"]).ljust(5), str(r["gejala_beda"]).ljust(5),
          str(round(r["jarak"], 3)).ljust(7), str(round(r["similarity"], 3)).ljust(11),
          round(r["jaccard"], 3))
    no += 1

# --- Langkah 3: diagnosis (Reuse) ---
r = reuse(hasil, len(kasus_baru))
print("\n" + "=" * 60)
print("HASIL CBR")
print("=" * 60)
print("Diagnosis :", label[r["diagnosis"]] if r["diagnosis"] else "Seri, perlu pakar")
print("Seri      :", r["seri"])
print("Peringatan:", r["peringatan"])

## 3. Revise dan Retain
Dokter boleh mengubah grade (Revise), lalu kasus disimpan ke basis (Retain).

In [ ]:
def retain(basis, x, grade):
    # cari nomor K terbesar yang sudah ada
    nomor_terbesar = 0
    for kasus in basis:
        if kasus["kode"].startswith("K"):
            nomor = int(kasus["kode"][1:])
            if nomor > nomor_terbesar:
                nomor_terbesar = nomor

    kasus_baru = {"kode": "K" + str(nomor_terbesar + 1), "grade": grade, "vektor": list(x)}
    return basis + [kasus_baru]          # basis lama tidak diubah


gejala_x = ["G1"]
r1 = cek_dbd(gejala_x, basis_kasus)
print("Usulan sistem :", r1["reuse"]["diagnosis"], r1["reuse"]["peringatan"])

grade_dokter = "II"                      # Revise: dokter mengubah grade
print("Grade dokter  :", grade_dokter)

basis_kasus2 = retain(basis_kasus, r1["x"], grade_dokter)   # Retain
print("Jumlah kasus sekarang:", len(basis_kasus2))

r2 = cek_dbd(gejala_x, basis_kasus2)     # Retrieve ulang
print("Setelah Retain:", r2["reuse"]["diagnosis"], "| jarak =", r2["reuse"]["d_min"])

## 4. Uji mandiri

In [ ]:
def jarak_per_grade(daftar_gejala):
    hasil = retrieve(ubah_ke_vektor(daftar_gejala), basis_kasus)
    jarak = {}
    for r in hasil:
        if r["grade"] not in jarak:
            jarak[r["grade"]] = round(r["jarak"], 3)
    return jarak

# 1. G1,G2,G8,G9,G12,G13,G14 -> DHF-II
c = cek_dbd(kasus_baru, basis_kasus)
assert c["reuse"]["diagnosis"] == "II" and not c["reuse"]["seri"]
assert jarak_per_grade(kasus_baru) == {"I": 1.414, "II": 1.0, "III": 1.414, "IV": 2.236}

# 2. G1,G8,G12,G13,G14 -> DHF-I (jarak = 0)
c = cek_dbd(["G1", "G8", "G12", "G13", "G14"], basis_kasus)
assert c["reuse"]["diagnosis"] == "I" and c["reuse"]["d_min"] == 0 and c["reuse"]["peringatan"] == []

# 3. G1,G6,G10,G12,G13,G14 -> DHF-IV (jarak = 0)
c = cek_dbd(["G1", "G6", "G10", "G12", "G13", "G14"], basis_kasus)
assert c["reuse"]["diagnosis"] == "IV" and c["reuse"]["d_min"] == 0

# 4. G1 saja -> peringatan "belum cukup spesifik"
c = cek_dbd(["G1"], basis_kasus)
assert any("belum cukup spesifik" in p for p in c["reuse"]["peringatan"])

# 5. Tanpa gejala -> pesan validasi
assert "error" in cek_dbd([], basis_kasus)

# 6. Seri: jarak sama ke DHF-II dan DHF-IV
c = cek_dbd(["G1", "G6", "G9", "G12", "G13", "G14"], basis_kasus)
assert c["reuse"]["seri"] and c["reuse"]["kandidat"] == ["II", "IV"] and c["reuse"]["diagnosis"] is None

# 7. Retain dipakai pada perhitungan berikutnya
assert cek_dbd(["G1"], basis_kasus2)["reuse"]["diagnosis"] == "II"

# 8. Peringatan jarak besar (> 2,5)
c = cek_dbd(["G2", "G3", "G4", "G5", "G7", "G11"], basis_kasus)
assert c["reuse"]["d_min"] > 2.5 and any("kurang cocok" in p for p in c["reuse"]["peringatan"])

print("Semua uji lulus")

## 5. Visualisasi sederhana (bar teks)

In [ ]:
hasil = retrieve(ubah_ke_vektor(kasus_baru), basis_kasus)
print("Kemiripan kasus contoh dengan tiap grade")
print("-" * 55)
for r in hasil:
    persen = r["similarity"] * 100
    batang = "█" * int(persen / 2)        # 1 blok = 2%
    print(label[r["grade"]].ljust(25), batang, str(round(persen, 1)) + "%")

## 6. Evaluasi (data dari artikel, bukan hasil model ini)
Confusion matrix dari Tabel 4-2 artikel Jatmiko dkk. (2017). Baris = pakar, kolom = sistem.

In [ ]:
kelas = ["DHF-I", "DHF-II", "DHF-III", "DHF-IV"]
cm = [
    [14, 2, 0, 0],
    [0,  5, 0, 0],
    [0,  0, 3, 0],
    [0,  0, 0, 1],
]

print("".ljust(14), *[("sistem " + k).ljust(14) for k in kelas])
for i in range(4):
    print(("pakar " + kelas[i]).ljust(14), *[str(v).ljust(14) for v in cm[i]])

total = 0
benar = 0
for i in range(4):
    for j in range(4):
        total += cm[i][j]
        if i == j:
            benar += cm[i][j]
print("\nAkurasi:", benar, "/", total, "=", round(benar / total * 100, 1), "%")

print("\nKelas      Precision  Recall")
for i in range(4):
    jumlah_kolom = 0
    jumlah_baris = 0
    for j in range(4):
        jumlah_kolom += cm[j][i]
        jumlah_baris += cm[i][j]
    print(kelas[i].ljust(10), str(round(cm[i][i] / jumlah_kolom, 3)).ljust(10), round(cm[i][i] / jumlah_baris, 3))